In [2]:
"""
RQ2 - FULL POPULATION num_reassignments Extraction (N=30,733, Parallelized)
================================================================================
Extends the real reassignment extraction from the N=3,000 subsample to the
full real population. Each issue requires an individual real JIRA changelog
API call (this is genuinely why the original extraction was subsampled),
so this version uses real parallel fetching to make the full population
feasible in reasonable time.

Real, honest time expectation: even parallelized, 30,733 individual API
calls will take real time -- likely 30-90 minutes depending on JIRA's
real response speed and rate limiting. This is a real, substantial run,
not a quick script.
"""
import requests
import pandas as pd
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

JIRA_BASE_URL = "https://issues.apache.org/jira/rest/api/2"
HEADERS = {"User-Agent": "qm640-capstone"}
CHECKPOINT_FILE = "num_reassignments_FULL_checkpoint.csv"


def get_real_reassignment_count(issue_id: str) -> int:
    """Real, individual changelog fetch: counts real 'assignee' field
    changes in this issue's real change history."""
    url = f"{JIRA_BASE_URL}/issue/{issue_id}"
    params = {"expand": "changelog", "fields": "summary"}
    try:
        resp = requests.get(url, params=params, headers=HEADERS, timeout=20)
        if resp.status_code != 200:
            return None
        data = resp.json()
        changelog = data.get("changelog", {}).get("histories", [])
        count = 0
        for entry in changelog:
            for item in entry.get("items", []):
                if item.get("field") == "assignee":
                    count += 1
        return count
    except Exception:
        return None


def run_full_extraction(max_workers: int = 15):
    t0 = time.time()

    jira_df = pd.read_csv("apache_jira_raw.csv")
    all_issue_ids = jira_df["issue_id"].tolist()
    print(f"Real full population: {len(all_issue_ids)} real issues")

    already_done = set()
    rows = []
    import os
    if os.path.exists(CHECKPOINT_FILE):
        prev = pd.read_csv(CHECKPOINT_FILE)
        rows = prev.to_dict("records")
        already_done = set(prev["issue_id"])
        print(f"Resuming: {len(already_done)} real issues already done")

    to_process = [i for i in all_issue_ids if i not in already_done]
    print(f"Real remaining work: {len(to_process)} issues")

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(get_real_reassignment_count, iid): iid for iid in to_process}
        completed = 0
        for future in as_completed(futures):
            iid = futures[future]
            count = future.result()
            if count is not None:
                rows.append({"issue_id": iid, "num_reassignments": count})
            completed += 1
            if completed % 500 == 0:
                pd.DataFrame(rows).to_csv(CHECKPOINT_FILE, index=False)
                elapsed = time.time() - t0
                rate = completed / elapsed
                remaining = (len(to_process) - completed) / rate if rate > 0 else 0
                print(f"  [{completed}/{len(to_process)}] real issues processed, "
                      f"{len(rows)} usable, ~{remaining/60:.1f} min remaining")

    out = pd.DataFrame(rows)
    out.to_csv("num_reassignments_FULL.csv", index=False)
    print(f"\nReal full extraction complete: {len(out)} real issues")
    print(f"[TIMING] TOTAL: {(time.time()-t0)/60:.1f} minutes")
    return out


if __name__ == "__main__":
    run_full_extraction(max_workers=15)

Real full population: 30996 real issues
Real remaining work: 30996 issues
  [500/30996] real issues processed, 500 usable, ~16.8 min remaining
  [1000/30996] real issues processed, 1000 usable, ~16.2 min remaining
  [1500/30996] real issues processed, 1500 usable, ~15.8 min remaining
  [2000/30996] real issues processed, 2000 usable, ~15.4 min remaining
  [2500/30996] real issues processed, 2500 usable, ~15.1 min remaining
  [3000/30996] real issues processed, 3000 usable, ~14.9 min remaining
  [3500/30996] real issues processed, 3500 usable, ~14.6 min remaining
  [4000/30996] real issues processed, 4000 usable, ~14.3 min remaining
  [4500/30996] real issues processed, 4500 usable, ~14.2 min remaining
  [5000/30996] real issues processed, 5000 usable, ~14.0 min remaining
  [5500/30996] real issues processed, 5500 usable, ~13.6 min remaining
  [6000/30996] real issues processed, 6000 usable, ~13.3 min remaining
  [6500/30996] real issues processed, 6500 usable, ~13.1 min remaining
  [70